In [1]:
import os
os.environ["HF_HOME"] = "/shared/data3/pk36/.cache"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
import sys

# Get the parent directory's path
parent_dir = os.path.dirname(os.getcwd())

# Add the parent directory to the system path
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

In [44]:
import glob
import json
import ast
from collections import defaultdict

In [3]:
directory_path = "../inspiration_pred_output/"
search_pattern = os.path.join(directory_path, '*.json')
json_files = glob.glob(search_pattern)

In [4]:
with open("../evaluation/processed_abstracts.json", "r") as f:
    groundtruth = json.load(f)

In [5]:
all_ids = {"_".join(key.split('_', maxsplit=2)[:2]):key for key in groundtruth.keys()}

In [6]:
len(all_ids)

1521

### Main Method

In [12]:
formatted_outputs = {}
invalid_files = []
count_valid, count_invalid = 0, 0
# Iterate through the list of file paths
for idx, file_path in enumerate(json_files):
    id = "_".join(os.path.basename(file_path).split("_", maxsplit=2)[:2])
    full_id = all_ids[id]
    # Open each file using a context manager
    with open(file_path, 'r', encoding='utf-8') as f:
        # Load the JSON data from the file
        data = json.load(f)
        # Check if valid output
        keys = [k for k in data.keys() if "idea_rankings" not in k]
        if (len(keys) > 4) and ("idea_rankings" in data) and (len(data["idea_rankings"]) >= 1):
            formatted_outputs[full_id] = {
                "research_problem": data["research_problem"],
                "target_domain": data["target_domain"],
                "target_domain_subfield": data["fine_grained_domain"],
                "predicted_takeaways": [{"source_domain": idea["source_domain"],
                                         "integration_mechanism": {
                                             "target_domain_elements": idea["idea_fragment"]["integration_mechanism"]["target_domain_elements"],
                                             "source_domain_takeaways": idea["idea_fragment"]["integration_mechanism"]["selected_takeaways"]
                                         }
                                         }
                                        for idea in data["idea_rankings"]],
                "gt_takeaways": {"source_domain":  groundtruth[full_id]["original_data"]["source_domain"],
                                 "integration_mechanism": groundtruth[full_id]["processed_abstract"]["integration_mechanism"]
                                 },
                "proposed_ideas": [{"source_domain": idea["source_domain"],
                                    "idea": idea["idea_fragment"]["concrete_realization"]
                                    } for idea in data["idea_rankings"]],
                "gt_idea": {"source_domain":  groundtruth[full_id]["original_data"]["source_domain"],
                            "idea": groundtruth[full_id]["processed_abstract"]["concrete_realization"]
                }

            }
            count_valid += 1
        else:
            invalid_files.append(full_id)
            count_invalid +=1

In [13]:
with open("final_mainmethod_results.json", "w") as f:
    json.dump(formatted_outputs, f, indent=2)

In [14]:
len(formatted_outputs)

400

### Baseline #1

In [21]:
with open("../evaluation/baseline_output/baseline1_direct.json", "r") as f:
    baseline_one = json.load(f)

In [22]:
formatted_baseone_outputs = {}
for id in formatted_outputs:
    baseline_data = baseline_one[id]
    formatted_baseone_outputs[id] = {
                "research_problem": baseline_data["research_problem"],
                "target_domain": formatted_outputs[id]["target_domain"],
                "target_domain_subfield": formatted_outputs[id]["target_domain_subfield"],
                "all_subfields": [idea["fine_grained_source_domain"] for idea in baseline_data["idea_rankings"]],
                "predicted_takeaways": [{"source_domain": idea["source_domain"],
                                         "integration_mechanism": {
                                             "target_domain_elements": idea["idea_fragment"]["integration_mechanism"]["target_domain_elements"],
                                             "source_domain_takeaways": idea["idea_fragment"]["integration_mechanism"]["source_domain_takeaways"]
                                         }
                                         }
                                        for idea in baseline_data["idea_rankings"]],
                "gt_takeaways": {"source_domain":  groundtruth[id]["original_data"]["source_domain"],
                                 "integration_mechanism": groundtruth[id]["processed_abstract"]["integration_mechanism"]
                                 },
                "proposed_ideas": [{"source_domain": idea["source_domain"],
                                    "idea": idea["idea_fragment"]["concrete_realization"]
                                    } for idea in baseline_data["idea_rankings"]],
                "gt_idea": {"source_domain":  groundtruth[id]["original_data"]["source_domain"],
                            "idea": groundtruth[id]["processed_abstract"]["concrete_realization"]
                }

            }

In [23]:
len(formatted_baseone_outputs)

400

In [24]:
with open("final_baseline_one_results.json", "w") as f:
    json.dump(formatted_baseone_outputs, f, indent=2)

### Baseline #2

In [25]:
with open("../evaluation/baseline_output/baseline2_dual_retrieval.json", "r") as f:
    baseline_two = json.load(f)

In [ ]:
formatted_basetwo_outputs = {}
for id in formatted_outputs:
    baseline_data = baseline_two[id]
    formatted_basetwo_outputs[id] = {
                "research_problem": baseline_data["research_problem"],
                "target_domain": formatted_outputs[id]["target_domain"],
                "target_domain_subfield": formatted_outputs[id]["target_domain_subfield"],
                "predicted_takeaways": [{"source_domain": idea["source_domain"],
                                         "integration_mechanism": {
                                             "target_domain_elements": idea["idea_fragment"]["integration_mechanism"]["target_domain_elements"],
                                             "source_domain_takeaways": idea["idea_fragment"]["integration_mechanism"]["source_domain_takeaways"]
                                         }
                                         }
                                        for idea in baseline_data["idea_rankings"]],
                "gt_takeaways": {"source_domain":  groundtruth[id]["original_data"]["source_domain"],
                                 "integration_mechanism": groundtruth[id]["processed_abstract"]["integration_mechanism"]
                                 },
                "proposed_ideas": [{"source_domain": idea["source_domain"],
                                    "idea": idea["idea_fragment"]["concrete_realization"]
                                    } for idea in baseline_data["idea_rankings"]],
                "gt_idea": {"source_domain":  groundtruth[id]["original_data"]["source_domain"],
                            "idea": groundtruth[id]["processed_abstract"]["concrete_realization"]
                }

            }

In [27]:
len(formatted_basetwo_outputs)

400

In [28]:
with open("final_baseline_two_results.json", "w") as f:
    json.dump(formatted_basetwo_outputs, f, indent=2)

## Results Analysis

In [ ]:
def win_rate_computation(method_refs, k):
    for eval_type, eval_results in method_refs.items():
        print(f"{eval_type} Evaluation:")
        for method_name, method_vals in eval_results.items():
            print(f"\t{method_name}:")
            stats = defaultdict(lambda: {"wins": 0, "ties": 0, "losses": 0})
            for key, value in method_vals.items():
                if key == "final_stats":
                    continue
                if type(value) != dict:
                    continue

                sample_id, idx = ast.literal_eval(key)
                if idx >= k:
                    continue

                for criteria in value[f"{eval_type}_comparison"].keys():
                    if value[f"{eval_type}_comparison"][criteria]['preferred_method'] == 1:
                        stats[criteria]["wins"] += 1
                    elif value[f"{eval_type}_comparison"][criteria]['preferred_method'] == 2:
                        stats[criteria]["losses"] += 1
                    else:
                        print(f"invalid {sample_id}")
                if value["overall_assessment"]["preferred_method"] == 1:
                    stats["overall"]["wins"] += 1
                elif value["overall_assessment"]["preferred_method"] == 2:
                    stats["overall"]["losses"] += 1
                else:
                    print(f"invalid {sample_id}")


            for key, value in stats.items():
                total = value["wins"] + value["ties"] + value["losses"]
                win_rate = value["wins"] / total if total > 0 else 0
                loss_rate = value["losses"] / total if total > 0 else 0
                print(f"\t\t{key}: Win Rate @ {k}: {win_rate:.2f}, Loss Rate: {loss_rate:.2f}")

In [57]:
with open("../evaluation/comparison/mainmethod_idea_eval.json") as f:
    mainmethod_idea_results = json.load(f)

with open("../evaluation/comparison/mainmethod_takeaway_eval.json") as f:
    mainmethod_takeaway_results = json.load(f)

with open("../evaluation/comparison/baseline_one_idea_eval.json") as f:
    baseline_one_idea_results = json.load(f)

with open("../evaluation/comparison/baseline_one_takeaway_eval.json") as f:
    baseline_one_takeaway_results = json.load(f)

with open("../evaluation/comparison/baseline_two_idea_eval.json") as f:
    baseline_two_idea_results = json.load(f)

with open("../evaluation/comparison/baseline_two_takeaway_eval.json") as f:
    baseline_two_takeaway_results = json.load(f)

In [48]:
with open("final_mainmethod_results.json", "r") as f:
    mainmethod_results = json.load(f)

with open("final_baseline_one_results.json", "r") as f:
    b_one_results = json.load(f)

with open("final_baseline_two_results.json", "r") as f:
    b_two_results = json.load(f)

In [56]:
for key, value in mainmethod_idea_results.items():
    if key == "final_stats":
        continue
    if type(value) != dict:
        continue

    sample_id, idx = ast.literal_eval(key)
    if idx >= 1:
        continue
    if (baseline_two_idea_results[key]["idea_comparison"]["usefulness"]["preferred_method"] == 1) and (value["idea_comparison"]["usefulness"]["preferred_method"] == 2):
        main_source = mainmethod_results[sample_id]["proposed_ideas"][idx]["source_domain"]
        btwo_source = b_two_results[sample_id]["proposed_ideas"][idx]["source_domain"]
        if main_source == btwo_source:
            print(sample_id, idx, main_source, btwo_source)

1799_11198_video_synthesis 0 Engineering Engineering
37209_33556_improve_models'_robustness_to_distributional_shifts 0 Psychology Psychology
489_29921_the_decision-making_problem 0 Psychology Psychology
33860_19347_individuals_who_use_myoelectric_upper-limb_prostheses 0 Psychology Psychology
4149_3861_machine_learning_models 0 Psychology Psychology
3708_5072_one-shot_detection 0 Psychology Psychology
8699_10270_applying_convolutional_neural_networks_to_spherical_images 0 Mathematics Mathematics
9795_1221_the_iris_network_design_process 0 Engineering Engineering
10162_1174_discover_the_common_and_salient_objects_from_a_group_of_relevant_images 0 Psychology Psychology
7485_328_learning_and_recognizing_objects_from_few_images 0 Psychology Psychology
32468_7482_token-to-expert_allocation 0 Operations Research Operations Research
3205_1594_object_detection 0 Psychology Psychology
17378_795_train_interpretable_convolutional_neural_networks 0 Physics Physics
15582_17524_both_image_embeddings_

In [61]:
# method_refs = {"Main Method": {"ideas": mainmethod_idea_results, "takeaway": mainmethod_takeaway_results},
#                "Baseline 1": {"ideas": baseline_one_idea_results, "takeaway": baseline_one_takeaway_results},
#                "Baseline 2": {"ideas": baseline_two_idea_results, "takeaway": baseline_two_takeaway_results}}

method_refs = {"idea": {"Main Method": mainmethod_idea_results,
                         "Baseline 1": baseline_one_idea_results,
                         "Baseline 2": baseline_two_idea_results
                         },
                "takeaway": {"Main Method": mainmethod_takeaway_results,
                             "Baseline 1": baseline_one_takeaway_results,
                             "Baseline 2": baseline_two_takeaway_results}}

In [60]:
# WITH TIE
win_rate_computation(method_refs=method_refs, k=1)

ideas Evaluation:
	Main Method:
		novelty: Win Rate @ 1: 0.85, Loss Rate: 0.15
		usefulness: Win Rate @ 1: 0.48, Loss Rate: 0.52
		overall: Win Rate @ 1: 0.57, Loss Rate: 0.42
	Baseline 1:
		novelty: Win Rate @ 1: 0.14, Loss Rate: 0.86
		usefulness: Win Rate @ 1: 0.69, Loss Rate: 0.32
		overall: Win Rate @ 1: 0.53, Loss Rate: 0.47
	Baseline 2:
		novelty: Win Rate @ 1: 0.70, Loss Rate: 0.29
		usefulness: Win Rate @ 1: 0.63, Loss Rate: 0.37
		overall: Win Rate @ 1: 0.70, Loss Rate: 0.30
takeaway Evaluation:
	Main Method:


KeyError: 'idea_comparison'

In [12]:
# WITH TIE
win_rate_computation(method_refs=method_refs, k=2)

ideas Evaluation:
	Main Method:
		novelty: Win Rate @ 2: 0.83, Tie Rate: 0.00, Loss Rate: 0.17
		usefulness: Win Rate @ 2: 0.49, Tie Rate: 0.00, Loss Rate: 0.51
		integration_quality: Win Rate @ 2: 0.81, Tie Rate: 0.00, Loss Rate: 0.19
		overall: Win Rate @ 2: 0.73, Tie Rate: 0.00, Loss Rate: 0.26
	Baseline 1:
		novelty: Win Rate @ 2: 0.21, Tie Rate: 0.00, Loss Rate: 0.79
		usefulness: Win Rate @ 2: 0.69, Tie Rate: 0.00, Loss Rate: 0.32
		integration_quality: Win Rate @ 2: 0.42, Tie Rate: 0.00, Loss Rate: 0.58
		overall: Win Rate @ 2: 0.44, Tie Rate: 0.00, Loss Rate: 0.55
	Baseline 2:
		novelty: Win Rate @ 2: 0.71, Tie Rate: 0.00, Loss Rate: 0.29
		usefulness: Win Rate @ 2: 0.59, Tie Rate: 0.00, Loss Rate: 0.41
		integration_quality: Win Rate @ 2: 0.77, Tie Rate: 0.00, Loss Rate: 0.23
		overall: Win Rate @ 2: 0.74, Tie Rate: 0.00, Loss Rate: 0.26
takeaway Evaluation:
	Main Method:
		rationale_alignment: Win Rate @ 2: 0.51, Tie Rate: 0.06, Loss Rate: 0.43
		integration_alignment: Win Ra

In [13]:
# NO TIE
win_rate_computation(method_refs=method_refs, k=1)

ideas Evaluation:
	Main Method:
		novelty: Win Rate @ 1: 0.83, Tie Rate: 0.00, Loss Rate: 0.17
		usefulness: Win Rate @ 1: 0.51, Tie Rate: 0.00, Loss Rate: 0.49
		integration_quality: Win Rate @ 1: 0.82, Tie Rate: 0.00, Loss Rate: 0.18
		overall: Win Rate @ 1: 0.75, Tie Rate: 0.00, Loss Rate: 0.25
	Baseline 1:
		novelty: Win Rate @ 1: 0.20, Tie Rate: 0.00, Loss Rate: 0.80
		usefulness: Win Rate @ 1: 0.70, Tie Rate: 0.00, Loss Rate: 0.30
		integration_quality: Win Rate @ 1: 0.40, Tie Rate: 0.00, Loss Rate: 0.60
		overall: Win Rate @ 1: 0.41, Tie Rate: 0.01, Loss Rate: 0.58
	Baseline 2:
		novelty: Win Rate @ 1: 0.68, Tie Rate: 0.00, Loss Rate: 0.33
		usefulness: Win Rate @ 1: 0.65, Tie Rate: 0.00, Loss Rate: 0.35
		integration_quality: Win Rate @ 1: 0.79, Tie Rate: 0.00, Loss Rate: 0.21
		overall: Win Rate @ 1: 0.77, Tie Rate: 0.00, Loss Rate: 0.23
takeaway Evaluation:
	Main Method:
		rationale_alignment: Win Rate @ 1: 0.50, Tie Rate: 0.06, Loss Rate: 0.44
		integration_alignment: Win Ra

In [14]:
# NO TIE
win_rate_computation(method_refs=method_refs, k=2)

ideas Evaluation:
	Main Method:
		novelty: Win Rate @ 2: 0.83, Tie Rate: 0.00, Loss Rate: 0.17
		usefulness: Win Rate @ 2: 0.49, Tie Rate: 0.00, Loss Rate: 0.51
		integration_quality: Win Rate @ 2: 0.81, Tie Rate: 0.00, Loss Rate: 0.19
		overall: Win Rate @ 2: 0.73, Tie Rate: 0.00, Loss Rate: 0.26
	Baseline 1:
		novelty: Win Rate @ 2: 0.21, Tie Rate: 0.00, Loss Rate: 0.79
		usefulness: Win Rate @ 2: 0.69, Tie Rate: 0.00, Loss Rate: 0.32
		integration_quality: Win Rate @ 2: 0.42, Tie Rate: 0.00, Loss Rate: 0.58
		overall: Win Rate @ 2: 0.44, Tie Rate: 0.00, Loss Rate: 0.55
	Baseline 2:
		novelty: Win Rate @ 2: 0.71, Tie Rate: 0.00, Loss Rate: 0.29
		usefulness: Win Rate @ 2: 0.59, Tie Rate: 0.00, Loss Rate: 0.41
		integration_quality: Win Rate @ 2: 0.77, Tie Rate: 0.00, Loss Rate: 0.23
		overall: Win Rate @ 2: 0.74, Tie Rate: 0.00, Loss Rate: 0.26
takeaway Evaluation:
	Main Method:
		rationale_alignment: Win Rate @ 2: 0.51, Tie Rate: 0.06, Loss Rate: 0.43
		integration_alignment: Win Ra

### Source Diversity

In [9]:
from collections import Counter
import plotly.express as px
import pandas as pd
from pydantic import BaseModel

In [4]:
from utils import batch_llm_inference
from vllm import LLM

/home/pk36/structured_survey/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 01-31 15:14:39 [__init__.py:216] Automatically detected platform cuda.


In [5]:
print("Loading model...")
llm = LLM(model="Qwen/Qwen3-14B", tensor_parallel_size=1)
print("Model loaded.\n")

Loading model...
INFO 01-31 15:14:44 [utils.py:233] non-default args: {'disable_log_stats': True, 'model': 'Qwen/Qwen3-14B'}


INFO 01-31 15:14:44 [model.py:547] Resolved architecture: Qwen3ForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 01-31 15:14:44 [model.py:1510] Using max model len 40960


2026-01-31 15:14:45,101	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 01-31 15:14:45 [scheduler.py:205] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=1218315) INFO 01-31 15:14:46 [core.py:644] Waiting for init message from front-end.
(EngineCore_DP0 pid=1218315) INFO 01-31 15:14:46 [core.py:77] Initializing a V1 LLM engine (v0.11.0) with config: model='Qwen/Qwen3-14B', speculative_config=None, tokenizer='Qwen/Qwen3-14B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=40960, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser=''), observability_config=ObservabilityConfig(show_hid

Loading safetensors checkpoint shards:   0% Completed | 0/8 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  12% Completed | 1/8 [00:00<00:02,  2.38it/s]
Loading safetensors checkpoint shards:  25% Completed | 2/8 [00:01<00:03,  1.75it/s]
Loading safetensors checkpoint shards:  38% Completed | 3/8 [00:01<00:03,  1.58it/s]
Loading safetensors checkpoint shards:  50% Completed | 4/8 [00:02<00:02,  1.37it/s]
Loading safetensors checkpoint shards:  62% Completed | 5/8 [00:03<00:02,  1.23it/s]
Loading safetensors checkpoint shards:  75% Completed | 6/8 [00:04<00:01,  1.14it/s]
Loading safetensors checkpoint shards:  88% Completed | 7/8 [00:05<00:00,  1.09it/s]
Loading safetensors checkpoint shards: 100% Completed | 8/8 [00:06<00:00,  1.05it/s]
Loading safetensors checkpoint shards: 100% Completed | 8/8 [00:06<00:00,  1.19it/s]
(EngineCore_DP0 pid=1218315) 


(EngineCore_DP0 pid=1218315) INFO 01-31 15:15:02 [default_loader.py:267] Loading weights took 7.03 seconds
(EngineCore_DP0 pid=1218315) INFO 01-31 15:15:03 [gpu_model_runner.py:2653] Model loading took 27.5185 GiB and 7.941663 seconds
(EngineCore_DP0 pid=1218315) INFO 01-31 15:15:15 [backends.py:548] Using cache directory: /home/pk36/.cache/vllm/torch_compile_cache/e45dec5b9c/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=1218315) INFO 01-31 15:15:15 [backends.py:559] Dynamo bytecode transform time: 11.34 s
(EngineCore_DP0 pid=1218315) INFO 01-31 15:15:21 [backends.py:164] Directly load the compiled graph(s) for dynamic shape from the cache, took 4.626 s
(EngineCore_DP0 pid=1218315) INFO 01-31 15:15:23 [monitor.py:34] torch.compile takes 11.34 s in total
(EngineCore_DP0 pid=1218315) INFO 01-31 15:15:26 [gpu_worker.py:298] Available KV cache memory: 13.64 GiB
(EngineCore_DP0 pid=1218315) INFO 01-31 15:15:26 [kv_cache_utils.py:1087] GPU KV cache size: 89,360 tokens
(Engin

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:08<00:00,  7.58it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:03<00:00,  9.53it/s]


(EngineCore_DP0 pid=1218315) INFO 01-31 15:15:40 [gpu_model_runner.py:3480] Graph capturing finished in 14 secs, took 0.94 GiB
(EngineCore_DP0 pid=1218315) INFO 01-31 15:15:40 [core.py:210] init engine (profile, create kv cache, warmup model) took 36.91 seconds
INFO 01-31 15:15:41 [llm.py:306] Supported_tasks: ['generate']
Model loaded.



In [6]:
def plot_target_dist(data, name="IdeaCatalyst"):
    # --- Collect source domains ---
    target_domains = []

    for sample in data.values():
        domain = sample.get("target_domain")
        target_domains.append(domain)

    # --- Count and sort domains ---
    domain_counts = Counter(target_domains)

    df = (
        pd.DataFrame(domain_counts.items(), columns=["Target Domain", "Count"])
        .sort_values("Count", ascending=False)
    )

    # --- Create bar plot ---
    fig = px.bar(
        df,
        x="Target Domain",
        y="Count",
        color="Target Domain",   # adds color variation
        text="Count",
        title=f"Distribution of Target Domains in {name}"
    )

    # --- Styling for modern look ---
    fig.update_traces(
        textposition="outside"
    )

    fig.update_layout(
        template="seaborn",
        height=600,              # increased height
        font=dict(size=14),
        title_font=dict(size=22),
        xaxis_title_font=dict(size=16),
        yaxis_title_font=dict(size=16),
        bargap=0.3,
        showlegend=False         # cleaner since color = domain
    )

    fig.update_yaxes(
        range=[0, df["Count"].max() * 1.15]  # prevents top cutoff
    )

    fig.show()

In [36]:
def plot_source_dist_multi(method_data_dict):
    """
    method_data_dict: dict
        key   -> method name (str)
        value -> data in the same format as before (dict or list)
    """

    records = []

    for method_name, data in method_data_dict.items():
        source_domains = []

        if type(data) != list:
            for sample in data.values():
                for idx, idea_entry in enumerate(sample.get("proposed_ideas", [])):
                    if idx >= 2:  # keep your top-2 constraint
                        break
                    domain = idea_entry.get("source_domain")
                    if domain is not None:
                        source_domains.append(domain)
        else:
            source_domains = data

        domain_counts = Counter(source_domains)

        for domain, count in domain_counts.items():
            if count < 10:
                continue
            records.append({
                "Method": method_name,
                "Source Domain": domain,
                "Count": count
            })

    # --- Create combined DataFrame ---
    df = pd.DataFrame(records)

    # Ensure consistent domain ordering (global sort)
    domain_order = (
        df.groupby("Source Domain")["Count"]
          .sum()
          .sort_values(ascending=False)
          .index
          .tolist()
    )

    # --- Plot grouped bar chart ---
    fig = px.bar(
        df,
        x="Source Domain",
        y="Count",
        color="Method",
        barmode="group",
        category_orders={"Source Domain": domain_order},
        title=f"Distribution of Source Domains Across Methods",
        text="Count"
    )

    # --- Styling ---
    fig.update_traces(
        textposition="outside"
    )

    fig.update_layout(
        template="seaborn",
        height=650,
        font=dict(size=14),
        title_font=dict(size=22),
        xaxis_title_font=dict(size=16),
        yaxis_title_font=dict(size=16),
        bargap=0.25,
        bargroupgap=0.1
    )

    fig.update_yaxes(
        range=[0, df["Count"].max() * 1.2]
    )

    fig.show()

In [29]:
def plot_source_dist(data, name="IdeaCatalyst"):
    if type(data) != list:
        # --- Collect source domains ---
        source_domains = []

        for sample in data.values():
            for idx, idea_entry in enumerate(sample.get("proposed_ideas", [])):
                if idx >= 2:
                    break
                domain = idea_entry.get("source_domain")
                if domain is not None:
                    source_domains.append(domain)
                
    else:
        source_domains = data

    # --- Count and sort domains ---
    domain_counts = Counter(source_domains)

    df = (
        pd.DataFrame(domain_counts.items(), columns=["Source Domain", "Count"])
        .sort_values("Count", ascending=False)
    )

    # --- Create bar plot ---
    fig = px.bar(
        df,
        x="Source Domain",
        y="Count",
        color="Source Domain",   # adds color variation
        text="Count",
        title=f"Distribution of Source Domains in {name}"
    )

    # --- Styling for modern look ---
    fig.update_traces(
        textposition="outside"
    )

    fig.update_layout(
        template="seaborn",
        height=600,              # increased height
        font=dict(size=14),
        title_font=dict(size=22),
        xaxis_title_font=dict(size=16),
        yaxis_title_font=dict(size=16),
        bargap=0.3,
        showlegend=False         # cleaner since color = domain
    )

    fig.update_yaxes(
        range=[0, df["Count"].max() * 1.15]  # prevents top cutoff
    )

    fig.show()

In [24]:
prompt = lambda fine_grained_domain : f"""Determine the most relevant **coarse-grained domain** (the target domain which encompasses the subfield, {fine_grained_domain}) that the problem best fits out of the following options (ONLY SELECT THE TARGET DOMAIN FROM THIS LIST):

Computer Science, Medicine, Chemistry, Biology, Materials Science, Physics, Geology, Psychology, Art, History, Geography, Sociology, Business, Political Science, Economics, Philosophy, Mathematics, Engineering, Environmental Science, Agricultural and Food Sciences, Education, Law, Linguistics

Output your answer in JSON format:
{{
    "coarse_grained_domain": [domain from above list]
}}

Subfield: {fine_grained_domain}

Output:

"""

class CoarseDomain(BaseModel):
    coarse_grained_domain: str

In [25]:
def convert_domains(data):
    raw_source_domains = []
    conversion_prompts = []
    for sample in data.values():
        for idea_entry in sample.get("proposed_ideas", []):
            domain = idea_entry.get("source_domain")
            if domain is not None:
                raw_source_domains.append(domain)
                conversion_prompts.append([{"role": "user", "content": prompt(domain)}])
    
    outputs = batch_llm_inference(llm, conversion_prompts, CoarseDomain.model_json_schema(), temperature=0, max_tokens=256)
    return [o["coarse_grained_domain"] for o in outputs]

In [26]:
plot_target_dist(mainmethod_results)

In [19]:
converted_domains = convert_domains(b_two_results)

Processed prompts: 100%|██████████| 1200/1200 [00:10<00:00, 113.07it/s, est. speed input: 15561.84 toks/s, output: 1673.73 toks/s]


In [38]:
plot_source_dist_multi({"Unguided RAG": b_one_results,
                        "Guided RAG": converted_domains,
                        "IdeaCatalyst": mainmethod_results})

In [30]:
plot_source_dist(mainmethod_results)

In [31]:
plot_source_dist(b_one_results, "Unguided Source Retrieval")

In [32]:
plot_source_dist(converted_domains, "Guided Source Retrieval")